In [1]:
# Importing Libraries

import pandas as pd
import numpy as np
import joblib

In [7]:
df = pd.read_csv('Customer-Churn.csv')

telco_data = df.copy()

telco_data.TotalCharges = pd.to_numeric(
    telco_data.TotalCharges,
    errors='coerce'
)

telco_data.dropna(how='any', inplace=True)

telco_data.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [8]:
bins = [0, 12, 24, 36, 48, 60, 72]

labels = [
    '1-12',
    '13-24',
    '25-36',
    '37-48',
    '49-60',
    '61-72'
]

telco_data['tenure_bin'] = pd.cut(
    telco_data['tenure'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

telco_data[['tenure', 'tenure_bin']].head()

,tenure,tenure_bin
0,1,1-12
1,34,25-36
2,2,1-12
3,45,37-48
4,2,1-12


In [9]:
X = telco_data.drop(
    columns=['customerID', 'Churn', 'tenure']
)

X = pd.get_dummies(
    X,
    drop_first=True
)

X.head()

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_bin_13-24,tenure_bin_25-36,tenure_bin_37-48,tenure_bin_49-60,tenure_bin_61-72
0,0,29.85,29.85,False,True,False,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
1,0,56.95,1889.50,True,False,False,True,False,False,False,...,False,False,False,False,True,False,True,False,False,False
2,0,53.85,108.15,True,False,False,True,False,False,False,...,False,True,False,False,True,False,False,False,False,False
3,0,42.30,1840.75,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,False,False,False,True,False,False,True,...,False,True,False,True,False,False,False,False,False,False


In [10]:
xgb_model = joblib.load(
    'best_xgboost_churn_model.pkl'
)

optuna_model = joblib.load(
    'best_optuna_churn_model.pkl'
)

In [11]:
scaler = joblib.load('churn_scaler.pkl')

In [12]:
X_scaled = scaler.transform(X)

In [13]:
scaler.transform(X)

array([[-0.44047021, -1.1613181 , -0.99189887, ..., -0.35164132,
        -0.36834135, -0.49276695],
       [-0.44047021, -0.26066216, -0.16962872, ..., -0.35164132,
        -0.36834135, -0.49276695],
       [-0.44047021, -0.36368922, -0.95727743, ..., -0.35164132,
        -0.36834135, -0.49276695],
       ...,
       [-0.44047021, -1.16962673, -0.85190977, ..., -0.35164132,
        -0.36834135, -0.49276695],
       [ 2.27030107,  0.3192805 , -0.86953   , ..., -0.35164132,
        -0.36834135, -0.49276695],
       [-0.44047021,  1.35785976,  2.02129355, ..., -0.35164132,
        -0.36834135,  2.02935688]])

In [14]:
xgb_prediction = xgb_model.predict(X_scaled)

optuna_prediction = optuna_model.predict(X_scaled)

In [15]:
xgb_probability = xgb_model.predict_proba(X_scaled)[:, 1]

optuna_probability = optuna_model.predict_proba(X_scaled)[:, 1]

In [16]:
prediction_df = telco_data[['customerID', 'Churn']].copy()

prediction_df['XGB_Churn_Probability'] = xgb_probability
prediction_df['XGB_Prediction'] = xgb_prediction

prediction_df['Optuna_Churn_Probability'] = optuna_probability
prediction_df['Optuna_Prediction'] = optuna_prediction

prediction_df.head()

,customerID,Churn,XGB_Churn_Probability,XGB_Prediction,Optuna_Churn_Probability,Optuna_Prediction
0,7590-VHVEG,No,0.839267,1,0.903778,1
1,5575-GNVDE,No,0.110895,0,0.043531,0
2,3668-QPYBK,Yes,0.650174,1,0.641488,1
3,7795-CFOCW,No,0.162614,0,0.104685,0
4,9237-HQITU,Yes,0.859189,1,0.871234,1


In [17]:
def risk_level(probability):
    if probability >= 0.70:
        return 'High Risk'
    elif probability >= 0.40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

In [18]:
prediction_df['XGB_Risk_Level'] = prediction_df['XGB_Churn_Probability'].apply(risk_level)

prediction_df['Optuna_Risk_Level'] = prediction_df['Optuna_Churn_Probability'].apply(risk_level)

prediction_df.head()

,customerID,Churn,XGB_Churn_Probability,XGB_Prediction,Optuna_Churn_Probability,Optuna_Prediction,XGB_Risk_Level,Optuna_Risk_Level
0,7590-VHVEG,No,0.839267,1,0.903778,1,High Risk,High Risk
1,5575-GNVDE,No,0.110895,0,0.043531,0,Low Risk,Low Risk
2,3668-QPYBK,Yes,0.650174,1,0.641488,1,Medium Risk,Medium Risk
3,7795-CFOCW,No,0.162614,0,0.104685,0,Low Risk,Low Risk
4,9237-HQITU,Yes,0.859189,1,0.871234,1,High Risk,High Risk


In [19]:
total_customers = len(telco_data)

churned_customers = (telco_data['Churn'] == 'Yes').sum()

churn_rate = (churned_customers / total_customers) * 100

high_risk_xgb = (prediction_df['XGB_Risk_Level'] == 'High Risk').sum()

high_risk_optuna = (prediction_df['Optuna_Risk_Level'] == 'High Risk').sum()

print("Total Customers:", total_customers)
print("Churned Customers:", churned_customers)
print("Overall Churn Rate:", round(churn_rate, 2), "%")
print("High Risk Customers - XGBoost:", high_risk_xgb)
print("High Risk Customers - Optuna:", high_risk_optuna)

Total Customers: 7032
Churned Customers: 1869
Overall Churn Rate: 26.58 %
High Risk Customers - XGBoost: 1587
High Risk Customers - Optuna: 1649


In [20]:
month_to_month = telco_data[
    telco_data['Contract'] == 'Month-to-month'
]

month_to_month_churn_rate = (
    month_to_month['Churn'].eq('Yes').mean() * 100
)


no_online_security = telco_data[
    telco_data['OnlineSecurity'] == 'No'
]

no_online_security_churn_rate = (
    no_online_security['Churn'].eq('Yes').mean() * 100
)


no_tech_support = telco_data[
    telco_data['TechSupport'] == 'No'
]

no_tech_support_churn_rate = (
    no_tech_support['Churn'].eq('Yes').mean() * 100
)


print(
    "Month-to-month churn rate:",
    round(month_to_month_churn_rate, 2), "%"
)

print(
    "No Online Security churn rate:",
    round(no_online_security_churn_rate, 2), "%"
)

print(
    "No Tech Support churn rate:",
    round(no_tech_support_churn_rate, 2), "%"
)

Month-to-month churn rate: 42.71 %
No Online Security churn rate: 41.78 %
No Tech Support churn rate: 41.65 %


In [21]:
business_insights = {
    'Total Customers': total_customers,
    'Churned Customers': churned_customers,
    'Overall Churn Rate': round(churn_rate, 2),
    'XGB High Risk Customers': high_risk_xgb,
    'Optuna High Risk Customers': high_risk_optuna,
    'Month-to-Month Churn Rate': round(month_to_month_churn_rate, 2),
    'No Online Security Churn Rate': round(no_online_security_churn_rate, 2),
    'No Tech Support Churn Rate': round(no_tech_support_churn_rate, 2)
}

business_insights

{'Total Customers': 7032,
 'Churned Customers': np.int64(1869),
 'Overall Churn Rate': np.float64(26.58),
 'XGB High Risk Customers': np.int64(1587),
 'Optuna High Risk Customers': np.int64(1649),
 'Month-to-Month Churn Rate': np.float64(42.71),
 'No Online Security Churn Rate': np.float64(41.78),
 'No Tech Support Churn Rate': np.float64(41.65)}

In [22]:
insights = []

# Overall churn
insights.append(
    f"The overall customer churn rate is {churn_rate:.2f}%."
)

# Month-to-month customers
if month_to_month_churn_rate > churn_rate:
    insights.append(
        f"Month-to-month customers have a higher churn rate "
        f"({month_to_month_churn_rate:.2f}%) than the overall churn rate."
    )

# Online Security
if no_online_security_churn_rate > churn_rate:
    insights.append(
        f"Customers without Online Security have a higher churn rate "
        f"({no_online_security_churn_rate:.2f}%) than the overall churn rate."
    )

# Tech Support
if no_tech_support_churn_rate > churn_rate:
    insights.append(
        f"Customers without Tech Support have a higher churn rate "
        f"({no_tech_support_churn_rate:.2f}%) than the overall churn rate."
    )

# XGBoost high-risk customers
insights.append(
    f"XGBoost identified {high_risk_xgb} customers as high-risk."
)

# Optuna high-risk customers
insights.append(
    f"Optuna identified {high_risk_optuna} customers as high-risk."
)

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

1. The overall customer churn rate is 26.58%.
2. Month-to-month customers have a higher churn rate (42.71%) than the overall churn rate.
3. Customers without Online Security have a higher churn rate (41.78%) than the overall churn rate.
4. Customers without Tech Support have a higher churn rate (41.65%) than the overall churn rate.
5. XGBoost identified 1587 customers as high-risk.
6. Optuna identified 1649 customers as high-risk.


In [23]:
recommendations = []

if month_to_month_churn_rate > churn_rate:
    recommendations.append(
        "Consider offering discounts or incentives to encourage "
        "month-to-month customers to move to longer-term contracts."
    )

if no_online_security_churn_rate > churn_rate:
    recommendations.append(
        "Consider promoting Online Security services to customers "
        "who currently do not have them."
    )

if no_tech_support_churn_rate > churn_rate:
    recommendations.append(
        "Consider offering Tech Support packages or targeted support "
        "to customers who currently do not have Tech Support."
    )

if high_risk_xgb > 0:
    recommendations.append(
        "Prioritize high-risk customers identified by the XGBoost model "
        "for retention campaigns."
    )

if high_risk_optuna > 0:
    recommendations.append(
        "Use the Optuna model's high-risk customer list to support "
        "targeted customer retention activities."
    )

print("BUSINESS RECOMMENDATIONS")
print("-" * 30)

for i, recommendation in enumerate(recommendations, 1):
    print(f"{i}. {recommendation}")

BUSINESS RECOMMENDATIONS
------------------------------
1. Consider offering discounts or incentives to encourage month-to-month customers to move to longer-term contracts.
2. Consider promoting Online Security services to customers who currently do not have them.
3. Consider offering Tech Support packages or targeted support to customers who currently do not have Tech Support.
4. Prioritize high-risk customers identified by the XGBoost model for retention campaigns.
5. Use the Optuna model's high-risk customer list to support targeted customer retention activities.


In [24]:
print("=" * 60)
print("AI CUSTOMER CHURN ANALYSIS")
print("=" * 60)

print(f"\nTotal Customers: {total_customers}")
print(f"Churned Customers: {churned_customers}")
print(f"Overall Churn Rate: {churn_rate:.2f}%")

print("\nAI-GENERATED INSIGHTS")
print("-" * 30)

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\nBUSINESS RECOMMENDATIONS")
print("-" * 30)

for i, recommendation in enumerate(recommendations, 1):
    print(f"{i}. {recommendation}")

AI CUSTOMER CHURN ANALYSIS

Total Customers: 7032
Churned Customers: 1869
Overall Churn Rate: 26.58%

AI-GENERATED INSIGHTS
------------------------------
1. The overall customer churn rate is 26.58%.
2. Month-to-month customers have a higher churn rate (42.71%) than the overall churn rate.
3. Customers without Online Security have a higher churn rate (41.78%) than the overall churn rate.
4. Customers without Tech Support have a higher churn rate (41.65%) than the overall churn rate.
5. XGBoost identified 1587 customers as high-risk.
6. Optuna identified 1649 customers as high-risk.

BUSINESS RECOMMENDATIONS
------------------------------
1. Consider offering discounts or incentives to encourage month-to-month customers to move to longer-term contracts.
2. Consider promoting Online Security services to customers who currently do not have them.
3. Consider offering Tech Support packages or targeted support to customers who currently do not have Tech Support.
4. Prioritize high-risk cust

In [25]:
prediction_df.to_csv(
    'churn_predictions.csv',
    index=False
)

print("Prediction file saved successfully!")

Prediction file saved successfully!


In [26]:
pd.read_csv('churn_predictions.csv').head(5)

,customerID,Churn,XGB_Churn_Probability,XGB_Prediction,Optuna_Churn_Probability,Optuna_Prediction,XGB_Risk_Level,Optuna_Risk_Level
0,7590-VHVEG,No,0.839267,1,0.903778,1,High Risk,High Risk
1,5575-GNVDE,No,0.110895,0,0.043531,0,Low Risk,Low Risk
2,3668-QPYBK,Yes,0.650174,1,0.641488,1,Medium Risk,Medium Risk
3,7795-CFOCW,No,0.162614,0,0.104685,0,Low Risk,Low Risk
4,9237-HQITU,Yes,0.859189,1,0.871234,1,High Risk,High Risk


In [27]:
import os

files = [
    'Customer-Churn.csv',
    'best_xgboost_churn_model.pkl',
    'best_optuna_churn_model.pkl',
    'churn_scaler.pkl',
    'churn_predictions.csv'
]

for file in files:
    if os.path.exists(file):
        print(file, "✓ Available")
    else:
        print(file, "✗ Missing")

Customer-Churn.csv ✓ Available
best_xgboost_churn_model.pkl ✓ Available
best_optuna_churn_model.pkl ✓ Available
churn_scaler.pkl ✓ Available
churn_predictions.csv ✓ Available
